# Fasal Nirnay — Price & Grade Prediction Pipeline
**Cell 1**: Setup (run once)  
**Cell 2**: Load → Merge → Feature Engineering → Save Parquet (run once)  
**Cell 3**: Load Parquet → Train both models (re-run anytime)

In [ ]:
import os, zipfile, io, warnings
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report,
)
warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────
WEATHER_DATASET_DIR = Path("/kaggle/input/datasets/mruddunijmodha/fasalnirnay")
MANDI_DATASET_DIR   = Path("/kaggle/input/daily-commodity-prices-india")
OUTPUT_DIR          = Path("/kaggle/working/pipeline_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Constants ─────────────────────────────────────────────────
TARGET_STATES = {"delhi", "rajasthan", "punjab", "haryana", "delhi ncr"}
STATE_MAP = {
    "delhi ncr": "Delhi", "delhi": "Delhi",
    "punjab": "Punjab", "haryana": "Haryana", "rajasthan": "Rajasthan",
}
MANDI_COLS = ["State", "District", "Market", "Commodity", "Variety",
              "Grade", "Arrival_Date", "Min_Price", "Commodity_Code"]
TRAIN_CUTOFF = pd.Timestamp("2022-01-01")

# ── Feature lists ─────────────────────────────────────────────
WEATHER_FEATS = [
    "temperature_mean","temperature_max","temperature_min",
    "humidity","precipitation","rainfall",
    "wind_speed","wind_direction",
    "soil_temperature_0_7cm","soil_moisture_0_7cm","solar_radiation",
    "temp_range","heat_stress","frost_risk","drought_flag","vpd","soil_stress"
]
CALENDAR_FEATS = [
    "year","month","quarter","week_of_year","day_of_year",
    "month_sin","month_cos","doy_sin","doy_cos"
]
CAT_FEATS = [
    "Commodity_enc","Variety_enc","season_enc",
    "weather_condition_enc","state_norm_enc","district_norm_enc"
]
LAG_FEATS = [
    "price_lag1","price_lag7","price_lag30",
    "price_roll7","price_roll14","price_roll30",
    "price_std7","price_std14","price_std30",
    "daily_listings"
]
PRICE_FEATS = WEATHER_FEATS + CALENDAR_FEATS + CAT_FEATS + LAG_FEATS
GRADE_FEATS  = WEATHER_FEATS + CALENDAR_FEATS + CAT_FEATS

print("Setup complete.")

## Data Pipeline
Run this cell **once** to load, merge, engineer features, and save the parquet.

In [ ]:
# ── Step 1: Load weather ─────────────────────────────────────
csv_path = sorted(WEATHER_DATASET_DIR.rglob("*.csv"))[0]
print(f"Weather file: {csv_path}")
weather = pd.read_csv(csv_path, parse_dates=["date"])
weather["state_lower"] = weather["state"].str.lower().str.strip()
weather = weather[weather["state_lower"].isin(TARGET_STATES)].copy()
weather["state_norm"]    = weather["state_lower"].map(STATE_MAP)
weather["district_norm"] = weather["location"].str.lower().str.strip()
print(f"Weather rows: {len(weather):,}")

# ── Step 2: Load mandi ────────────────────────────────────────
def _read_csv_safe(f, name):
    try:
        return pd.read_csv(f, usecols=MANDI_COLS, parse_dates=["Arrival_Date"],
                           dayfirst=True, low_memory=False)
    except Exception as e:
        print(f"  Skipping {name}: {e}")
        return None

frames = []
all_files = sorted(MANDI_DATASET_DIR.rglob("*"))
csv_files = [f for f in all_files if f.suffix.lower() == ".csv" and f.is_file()]
zip_files = [f for f in all_files if f.suffix.lower() == ".zip" and f.is_file()]
print(f"Found {len(csv_files)} CSV(s) and {len(zip_files)} ZIP(s)")

for p in csv_files:
    with open(p, "rb") as f:
        chunk = _read_csv_safe(io.TextIOWrapper(f, encoding="utf-8", errors="replace"), p.name)
        if chunk is not None:
            frames.append(chunk)
            print(f"  {p.name}: {len(chunk):,} rows")

for zp in zip_files:
    with zipfile.ZipFile(zp) as zf:
        for name in [n for n in zf.namelist() if n.lower().endswith(".csv")]:
            with zf.open(name) as raw:
                chunk = _read_csv_safe(io.TextIOWrapper(raw, encoding="utf-8", errors="replace"), name)
                if chunk is not None:
                    frames.append(chunk)
                    print(f"  {name}: {len(chunk):,} rows")

mandi = pd.concat(frames, ignore_index=True)
mandi["state_lower"] = mandi["State"].str.lower().str.strip()
mandi = mandi[mandi["state_lower"].isin(TARGET_STATES)].copy()
mandi["state_norm"]    = mandi["state_lower"].map(STATE_MAP)
mandi["district_norm"] = mandi["District"].str.lower().str.strip()
mandi = mandi.dropna(subset=["Min_Price","Arrival_Date"])
mandi = mandi[mandi["Min_Price"] > 0]
print(f"Mandi rows: {len(mandi):,}")

# ── Step 3: Merge ─────────────────────────────────────────────
weather_key = weather[["date","state_norm","district_norm",
    "temperature_mean","temperature_max","temperature_min",
    "humidity","precipitation","rainfall","wind_speed","wind_direction",
    "soil_temperature_0_7cm","soil_moisture_0_7cm",
    "solar_radiation","weather_condition","season"]].copy()
weather_key.rename(columns={"district_norm":"w_loc"}, inplace=True)

mandi["date"] = pd.to_datetime(mandi["Arrival_Date"]).dt.normalize()

merged_exact = mandi.merge(weather_key,
    left_on=["date","state_norm","district_norm"],
    right_on=["date","state_norm","w_loc"], how="left")

unmatched_mask = merged_exact["temperature_mean"].isna()
matched_df     = merged_exact[~unmatched_mask].copy()
unmatched_df   = merged_exact[ unmatched_mask].copy()
print(f"Exact matches: {len(matched_df):,}  |  Fallback: {len(unmatched_df):,}")

drop_cols = ["temperature_mean","temperature_max","temperature_min",
    "humidity","precipitation","rainfall","wind_speed","wind_direction",
    "soil_temperature_0_7cm","soil_moisture_0_7cm","solar_radiation",
    "weather_condition","season","w_loc"]
unmatched_df = unmatched_df.drop(columns=drop_cols, errors="ignore")

weather_state_day = (weather_key
    .groupby(["date","state_norm"])[["temperature_mean","temperature_max",
        "temperature_min","humidity","precipitation","rainfall",
        "wind_speed","wind_direction","soil_temperature_0_7cm",
        "soil_moisture_0_7cm","solar_radiation"]].mean().reset_index())
mode_extra = (weather_key.groupby(["date","state_norm"])[["season","weather_condition"]]
    .agg(lambda x: x.mode().iloc[0] if len(x) > 0 else np.nan).reset_index())
weather_state_day = weather_state_day.merge(mode_extra, on=["date","state_norm"], how="left")

fallback = unmatched_df.merge(weather_state_day, on=["date","state_norm"], how="left")
merged = pd.concat([matched_df, fallback], ignore_index=True)
merged = merged.dropna(subset=["temperature_mean"])
print(f"Final merged rows: {len(merged):,}")

# ── Step 4: Feature engineering ──────────────────────────────
merged["year"]         = merged["date"].dt.year
merged["month"]        = merged["date"].dt.month
merged["day_of_year"]  = merged["date"].dt.dayofyear
merged["week_of_year"] = merged["date"].dt.isocalendar().week.astype(int)
merged["quarter"]      = merged["date"].dt.quarter
merged["month_sin"] = np.sin(2*np.pi*merged["month"]/12)
merged["month_cos"] = np.cos(2*np.pi*merged["month"]/12)
merged["doy_sin"]   = np.sin(2*np.pi*merged["day_of_year"]/365)
merged["doy_cos"]   = np.cos(2*np.pi*merged["day_of_year"]/365)
merged["temp_range"]   = merged["temperature_max"] - merged["temperature_min"]
merged["heat_stress"]  = (merged["temperature_max"] > 40).astype(int)
merged["frost_risk"]   = (merged["temperature_min"] <  4).astype(int)
merged["drought_flag"] = ((merged["rainfall"] < 2) & (merged["humidity"] < 30)).astype(int)
merged["vpd"] = ((1 - merged["humidity"]/100) * 0.6108
    * np.exp(17.27*merged["temperature_mean"]/(merged["temperature_mean"]+237.3)))
merged["soil_stress"] = (merged["soil_temperature_0_7cm"]*0.5
    + (1 - merged["soil_moisture_0_7cm"].clip(0,1))*0.5)

le_dict = {}
for col in ["Commodity","Variety","season","weather_condition","state_norm","district_norm"]:
    le = LabelEncoder()
    merged[col+"_enc"] = le.fit_transform(merged[col].astype(str).fillna("Unknown"))
    le_dict[col] = le

merged = merged.sort_values(["Commodity","state_norm","date"]).reset_index(drop=True)
for w in [7,14,30]:
    merged[f"price_roll{w}"] = (merged.groupby(["Commodity","state_norm"])["Min_Price"]
        .transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean()))
    merged[f"price_std{w}"]  = (merged.groupby(["Commodity","state_norm"])["Min_Price"]
        .transform(lambda x: x.shift(1).rolling(w, min_periods=1).std().fillna(0)))
merged["price_lag1"]  = merged.groupby(["Commodity","state_norm"])["Min_Price"].shift(1)
merged["price_lag7"]  = merged.groupby(["Commodity","state_norm"])["Min_Price"].shift(7)
merged["price_lag30"] = merged.groupby(["Commodity","state_norm"])["Min_Price"].shift(30)
merged["daily_listings"] = (merged.groupby(["date","Commodity","state_norm"])["Min_Price"]
    .transform("count"))

# ── Save ──────────────────────────────────────────────────────
joblib.dump(le_dict, OUTPUT_DIR / "label_encoders.pkl")
merged.to_parquet(OUTPUT_DIR / "merged_engineered.parquet", index=False)
print(f"Saved parquet + encoders → {OUTPUT_DIR}")

## Model Training
Run this cell directly if  already exists — skips all the heavy preprocessing.

In [ ]:
# ── Load saved parquet ───────────────────────────────────────
print("Loading parquet …")
df = pd.read_parquet(OUTPUT_DIR / "merged_engineered.parquet")
df["date"] = pd.to_datetime(df["date"])
print(f"Loaded {len(df):,} rows")

# ════════════════════════════════════════════════════════════
# MODEL 1 — PRICE PREDICTION  (LightGBM Regressor)
# ════════════════════════════════════════════════════════════
print("
[1/2] Training Price model …")
sub = df.dropna(subset=PRICE_FEATS + ["Min_Price"]).copy()
if len(sub) > 5_000_000:
    sub = sub.sample(5_000_000, random_state=42)
    print("  Sampled to 5M rows")

X, y = sub[PRICE_FEATS], sub["Min_Price"]
mask = sub["date"] < TRAIN_CUTOFF
X_train, X_val = X[mask], X[~mask]
y_train, y_val = y[mask], y[~mask]
print(f"  Train: {len(X_train):,}  |  Val: {len(X_val):,}")

price_imp = SimpleImputer(strategy="median")
X_train   = price_imp.fit_transform(X_train)
X_val     = price_imp.transform(X_val)

price_model = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05, num_leaves=127,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
    n_jobs=-1, random_state=42, verbose=-1
)
price_model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)])

preds = price_model.predict(X_val)
print(f"
  MAE  : ₹{mean_absolute_error(y_val, preds):.2f}")
print(f"  RMSE : ₹{mean_squared_error(y_val, preds)**0.5:.2f}")
print(f"  R²   :  {r2_score(y_val, preds):.4f}")
print(f"  MAPE :  {np.mean(np.abs((y_val-preds)/y_val.clip(1)))*100:.2f}%")

joblib.dump(price_model, OUTPUT_DIR / "price_model.pkl")
joblib.dump(price_imp,   OUTPUT_DIR / "price_imputer.pkl")
print("  Saved price_model.pkl")

# ════════════════════════════════════════════════════════════
# MODEL 2 — GRADE PREDICTION  (LightGBM Classifier)
# ════════════════════════════════════════════════════════════
print("
[2/2] Training Grade model …")
sub = df.dropna(subset=GRADE_FEATS + ["Grade"]).copy()
if len(sub) > 5_000_000:
    sub = sub.sample(5_000_000, random_state=42)
    print("  Sampled to 5M rows")

le_grade = LabelEncoder()
sub["Grade_enc"] = le_grade.fit_transform(sub["Grade"].astype(str).fillna("Unknown"))
print(f"  Grade classes: {le_grade.classes_.tolist()}")

X, y = sub[GRADE_FEATS], sub["Grade_enc"]
mask = sub["date"] < TRAIN_CUTOFF
X_train, X_val = X[mask], X[~mask]
y_train, y_val = y[mask], y[~mask]
print(f"  Train: {len(X_train):,}  |  Val: {len(X_val):,}")

grade_imp = SimpleImputer(strategy="most_frequent")
X_train   = grade_imp.fit_transform(X_train)
X_val     = grade_imp.transform(X_val)

grade_model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=127,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
    class_weight="balanced", n_jobs=-1, random_state=42, verbose=-1
)
grade_model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)])

preds = grade_model.predict(X_val)
print(f"
  Accuracy : {accuracy_score(y_val, preds)*100:.2f}%")
print(classification_report(y_val, preds, target_names=le_grade.classes_, zero_division=0))

joblib.dump(grade_model, OUTPUT_DIR / "grade_model.pkl")
joblib.dump(grade_imp,   OUTPUT_DIR / "grade_imputer.pkl")
joblib.dump(le_grade,    OUTPUT_DIR / "grade_label_encoder.pkl")
print("  Saved grade_model.pkl")

print("
" + "="*50)
print("  All models saved to", OUTPUT_DIR)
print("="*50)